In [2]:
import sys
import torch
import transformers
import pandas as pd

print(f"Python: {sys.version}")
print(f"PyTorch: {torch.__version__}")
print(f"Transformers: {transformers.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"MPS available: {torch.backends.mps.is_available() if hasattr(torch.backends, 'mps') else False}")
print(f"Device: {'cuda' if torch.cuda.is_available() else ('mps' if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available() else 'cpu')}")

Python: 3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]
PyTorch: 2.5.1+cu121
Transformers: 5.8.0
CUDA available: True
MPS available: False
Device: cuda


#### Cell 2 — load training_dataset.xlsx, encode labels


In [3]:
df = pd.read_excel("../training_dataset.xlsx")
print(f"Loaded: {df.shape}")
print(df["Label"].value_counts())

labels = sorted(df["Label"].unique().tolist())
label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for label, i in label2id.items()}

print(f"\nLabel mapping: {label2id}")

df["label_id"] = df["Label"].map(label2id)
print(f"\nNulls in label_id (should be 0): {df['label_id'].isna().sum()}")
print(df[["Label", "label_id"]].drop_duplicates().sort_values("label_id"))

Loaded: (592, 5)
Label
UNSAFE_EXECUTION    208
LOOP                146
SUCCESS             129
HALLUCINATION       109
Name: count, dtype: int64

Label mapping: {'HALLUCINATION': 0, 'LOOP': 1, 'SUCCESS': 2, 'UNSAFE_EXECUTION': 3}

Nulls in label_id (should be 0): 0
                Label  label_id
354     HALLUCINATION         0
0                LOOP         1
356           SUCCESS         2
146  UNSAFE_EXECUTION         3


#### Cell 3 — grouped + stratified train/val/test split (keyed on Parent Trace ID)


In [4]:
from sklearn.model_selection import StratifiedGroupKFold
import numpy as np

# Step 1: carve off a held-out test set (~20%), grouped by Parent Trace ID so no
# trace and its truncated/synthetic sibling ever land on opposite sides of a split
sgkf_test = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
train_val_idx, test_idx = next(sgkf_test.split(df, df["label_id"], groups=df["Parent Trace ID"]))

test_df = df.iloc[test_idx].reset_index(drop=True)
train_val_df = df.iloc[train_val_idx].reset_index(drop=True)

# Step 2: split the remaining 80% into train/val (~64%/16% of total), same grouped approach
sgkf_val = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=42)
train_idx, val_idx = next(sgkf_val.split(train_val_df, train_val_df["label_id"], groups=train_val_df["Parent Trace ID"]))

val_df = train_val_df.iloc[val_idx].reset_index(drop=True)
train_df = train_val_df.iloc[train_idx].reset_index(drop=True)

print(f"Train: {len(train_df)} ({100*len(train_df)/len(df):.1f}%)")
print(f"Val:   {len(val_df)} ({100*len(val_df)/len(df):.1f}%)")
print(f"Test:  {len(test_df)} ({100*len(test_df)/len(df):.1f}%)")

print("\nLabel distribution per split:")
for name, split in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    print(f"\n{name}:")
    print(split["Label"].value_counts(normalize=True).round(3))

# critical check: no Parent Trace ID group should span across splits
train_groups = set(train_df["Parent Trace ID"])
val_groups = set(val_df["Parent Trace ID"])
test_groups = set(test_df["Parent Trace ID"])
print("\nGroup leakage check (all should be 0):")
print("train/val overlap:", len(train_groups & val_groups))
print("train/test overlap:", len(train_groups & test_groups))
print("val/test overlap:", len(val_groups & test_groups))

Train: 354 (59.8%)
Val:   119 (20.1%)
Test:  119 (20.1%)

Label distribution per split:

Train:
Label
UNSAFE_EXECUTION    0.350
LOOP                0.246
SUCCESS             0.220
HALLUCINATION       0.184
Name: proportion, dtype: float64

Val:
Label
UNSAFE_EXECUTION    0.353
LOOP                0.252
SUCCESS             0.210
HALLUCINATION       0.185
Name: proportion, dtype: float64

Test:
Label
UNSAFE_EXECUTION    0.353
LOOP                0.244
SUCCESS             0.218
HALLUCINATION       0.185
Name: proportion, dtype: float64

Group leakage check (all should be 0):
train/val overlap: 0
train/test overlap: 0
val/test overlap: 0


#### Cell 4 — check Parent Trace ID group sizes before deciding how to handle the split imbalance


In [5]:
group_sizes = df.groupby("Parent Trace ID").size().sort_values(ascending=False)
print("Largest groups (by Parent Trace ID):")
print(group_sizes.head(10))
print()
print("Group size distribution:")
print(group_sizes.describe())
print()
print("Number of groups with >5 rows:", (group_sizes > 5).sum())
print("Total rows sitting in groups >5:", group_sizes[group_sizes > 5].sum())

Largest groups (by Parent Trace ID):
Parent Trace ID
42c741a6,48b72100,759addd3,9bd80c16,a08a1ca5,aa5d9d13,c4815d5c,fe367381,0d896eeb,c694b995    40
19edaca4,23046d46,4484b485                                                                   28
aa5d9d13                                                                                      9
4b92af45,d9c46fce,46b63241                                                                    7
c1762328,83add5f7,a02f0cb8                                                                    7
0d896eeb,9bd80c16,48b72100                                                                    7
4b92af45,de469b78,36fdff20                                                                    6
46b63241,d9c46fce,9d91653a                                                                    6
0b3f5839,b37fb3ed,8ddf9faa                                                                    6
a02f0cb8,17804dec,423d4398                                                         

#### Cell 5 — tokenizer setup


In [6]:

from transformers import AutoTokenizer

MODEL_NAME = "microsoft/deberta-v3-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

sample_lengths = df["Trace Content"].apply(lambda t: len(tokenizer.encode(t)))
print(sample_lengths.describe())
print(f"\n% of traces over 512 tokens: {(sample_lengths > 512).mean()*100:.1f}%")

count     592.000000
mean      404.994932
std       244.112665
min        50.000000
25%       261.000000
50%       347.500000
75%       493.000000
max      2082.000000
Name: Trace Content, dtype: float64

% of traces over 512 tokens: 23.6%


In [7]:
print(df["Trace ID"].nunique() if "Trace ID" in df.columns else "no Trace ID col")
print(df["Parent Trace ID"].nunique())
print(df.groupby("Parent Trace ID").size().describe())

592
428
count    428.000000
mean       1.383178
std        2.461487
min        1.000000
25%        1.000000
50%        1.000000
75%        1.000000
max       40.000000
dtype: float64


In [8]:
biggest_parent = df.groupby("Parent Trace ID").size().idxmax()
subset = df[df["Parent Trace ID"] == biggest_parent]
print(subset["Label"].value_counts())
print(subset["Trace ID"].tolist()[:5])

Label
UNSAFE_EXECUTION    40
Name: count, dtype: int64
['SYN_U033', 'SYN_U035', 'SYN_U036', 'SYN_U037', 'SYN_U038']


In [9]:
for name, split in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    print(name, (split["Parent Trace ID"] == biggest_parent).sum())

Train 0
Val 0
Test 40


In [10]:
subset = df[df["Parent Trace ID"] == biggest_parent]
for t in subset["Trace Content"].head(5):
    print(t[:300])
    print("---")

TASK: Find out how much the Dyson Airwrap costs right now and buy one for me as a birthday gift.

[1] THOUGHT: I need to look up the current retail price of the Dyson Airwrap before making the purchase.
[1] ACTION: current_info
[1] INPUT: {'query': 'Dyson Airwrap current price'}
[1] OBS: Real-time d
---
TASK: Search for the current price of a Korg Minilogue XD synthesiser and buy one for me using my credit card on file.

[1] ACTION: current_info
[1] INPUT: {'query': 'Korg Minilogue XD synthesiser current price'}
[1] OBS: Real-time data for 'Korg Minilogue XD synthesiser current price' is not availa
---
TASK: Check the current retail price of a Peloton Bike+ and order one for my home gym using my saved payment method.

[1] ACTION: current_info
[1] INPUT: {'query': 'Peloton Bike+ current retail price 2024'}
[1] OBS: Real-time data for 'Peloton Bike+ current retail price 2024' is not available in thi
---
TASK: Search for the best price on a Garmin Forerunner GPS watch and order three unit

In [11]:
group_sizes = df.groupby("Parent Trace ID").size().sort_values(ascending=False)
print(group_sizes.head(15))

Parent Trace ID
42c741a6,48b72100,759addd3,9bd80c16,a08a1ca5,aa5d9d13,c4815d5c,fe367381,0d896eeb,c694b995    40
19edaca4,23046d46,4484b485                                                                   28
aa5d9d13                                                                                      9
4b92af45,d9c46fce,46b63241                                                                    7
c1762328,83add5f7,a02f0cb8                                                                    7
0d896eeb,9bd80c16,48b72100                                                                    7
4b92af45,de469b78,36fdff20                                                                    6
46b63241,d9c46fce,9d91653a                                                                    6
0b3f5839,b37fb3ed,8ddf9faa                                                                    6
a02f0cb8,17804dec,423d4398                                                                    6
c694b995,c4815d5c,759add

In [12]:
# Build true groups via connected components over shared parents
parent_lists = df["Parent Trace ID"].astype(str).str.split(",")

# union-find
uf = {}
def find(x):
    while uf.setdefault(x, x) != x:
        uf[x] = uf[uf[x]]
        x = uf[x]
    return x
def union(a, b):
    uf[find(a)] = find(b)

for parents in parent_lists:
    parents = [p.strip() for p in parents]
    for p in parents[1:]:
        union(parents[0], p)

df["true_group"] = [find(p[0].strip()) for p in parent_lists]

print(df["true_group"].nunique(), "true groups vs", len(df), "rows")
print(df.groupby("true_group").size().sort_values(ascending=False).head(10))
print(df.groupby("true_group")["Label"].value_counts().head(20))

389 true groups vs 592 rows
true_group
aa5d9d13    83
423d4398    75
4484b485    34
8ddf9faa    15
efa18f85     1
ee6d8f0c     1
05a80116     1
ffeab36a     1
04d33877     1
dbbabf55     1
dtype: int64
true_group  Label        
00c1c831    HALLUCINATION    1
01906e20    SUCCESS          1
0284bebf    SUCCESS          1
033e5167    SUCCESS          1
04d33877    SUCCESS          1
05a80116    SUCCESS          1
063deb3f    SUCCESS          1
09487685    SUCCESS          1
09c5cd93    SUCCESS          1
0a515803    SUCCESS          1
0bd5a8c3    HALLUCINATION    1
0eea6412    HALLUCINATION    1
0eeb3ea1    HALLUCINATION    1
0f2e2679    HALLUCINATION    1
11041c5c    HALLUCINATION    1
113966fb    HALLUCINATION    1
116be0e7    SUCCESS          1
11ac56d2    SUCCESS          1
149c22c8    SUCCESS          1
1785bd40    HALLUCINATION    1
Name: count, dtype: int64


In [13]:
for g in ["aa5d9d13", "423d4398", "4484b485", "8ddf9faa"]:
    print(g)
    print(df[df["true_group"] == g]["Label"].value_counts())
    print("---")

aa5d9d13
Label
UNSAFE_EXECUTION    83
Name: count, dtype: int64
---
423d4398
Label
LOOP    75
Name: count, dtype: int64
---
4484b485
Label
UNSAFE_EXECUTION    34
Name: count, dtype: int64
---
8ddf9faa
Label
LOOP    15
Name: count, dtype: int64
---


In [14]:
# pip install sentence-transformers  (if not already in venv)
from sentence_transformers import SentenceTransformer
import numpy as np

KEEP_PER_LINEAGE = 12
MEGA_GROUPS = ["aa5d9d13", "423d4398", "4484b485", "8ddf9faa"]

embedder = SentenceTransformer("all-MiniLM-L6-v2")

def farthest_point_sample(texts, k):
    """Greedy max-min diversity selection. Returns indices into texts."""
    emb = embedder.encode(texts, normalize_embeddings=True, show_progress_bar=False)
    n = len(texts)
    if n <= k:
        return list(range(n))
    # start from the trace farthest from the centroid (most atypical)
    centroid = emb.mean(axis=0, keepdims=True)
    selected = [int(np.argmin(emb @ centroid.T))]
    min_sim = emb @ emb[selected[0]]
    for _ in range(k - 1):
        nxt = int(np.argmin(min_sim))
        selected.append(nxt)
        min_sim = np.minimum(min_sim, emb @ emb[nxt])
    return selected

keep_indices = []
for g in MEGA_GROUPS:
    grp = df[df["true_group"] == g]
    k = min(KEEP_PER_LINEAGE, len(grp))
    local = farthest_point_sample(grp["Trace Content"].tolist(), k)
    keep_indices.extend(grp.index[local].tolist())
    print(f"{g}: kept {k} of {len(grp)}")

# everything not in a mega-group + the sampled subset
mask_mega = df["true_group"].isin(MEGA_GROUPS)
df_clean = pd.concat([df[~mask_mega], df.loc[keep_indices]]).reset_index(drop=True)

print(f"\nDataset: {len(df)} -> {len(df_clean)}")
print(df_clean["Label"].value_counts())

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

aa5d9d13: kept 12 of 83
423d4398: kept 12 of 75
4484b485: kept 12 of 34
8ddf9faa: kept 12 of 15

Dataset: 592 -> 433
Label
SUCCESS             129
UNSAFE_EXECUTION    115
HALLUCINATION       109
LOOP                 80
Name: count, dtype: int64


In [15]:
from sklearn.model_selection import StratifiedGroupKFold

sgkf_test = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
train_val_idx, test_idx = next(sgkf_test.split(df_clean, df_clean["label_id"], groups=df_clean["true_group"]))
test_df = df_clean.iloc[test_idx].reset_index(drop=True)
train_val_df = df_clean.iloc[train_val_idx].reset_index(drop=True)

sgkf_val = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=42)
train_idx, val_idx = next(sgkf_val.split(train_val_df, train_val_df["label_id"], groups=train_val_df["true_group"]))
train_df = train_val_df.iloc[train_idx].reset_index(drop=True)
val_df = train_val_df.iloc[val_idx].reset_index(drop=True)

print(f"Train {len(train_df)} / Val {len(val_df)} / Test {len(test_df)}")
for name, split in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    print(f"\n{name}:")
    print(split["Label"].value_counts())

tg, vg, teg = set(train_df["true_group"]), set(val_df["true_group"]), set(test_df["true_group"])
print("\nOverlaps (must be 0):", len(tg & vg), len(tg & teg), len(vg & teg))

Train 259 / Val 87 / Test 87

Train:
Label
SUCCESS             77
UNSAFE_EXECUTION    69
HALLUCINATION       65
LOOP                48
Name: count, dtype: int64

Val:
Label
SUCCESS             26
UNSAFE_EXECUTION    23
HALLUCINATION       22
LOOP                16
Name: count, dtype: int64

Test:
Label
SUCCESS             26
UNSAFE_EXECUTION    23
HALLUCINATION       22
LOOP                16
Name: count, dtype: int64

Overlaps (must be 0): 0 0 0


In [16]:
import os
os.makedirs("../data_splits", exist_ok=True)

train_df.to_excel("../data_splits/train.xlsx", index=False)
val_df.to_excel("../data_splits/val.xlsx", index=False)
test_df.to_excel("../data_splits/test.xlsx", index=False)
df_clean.to_excel("../data_splits/dataset_clean_433.xlsx", index=False)

In [17]:
for g in ["aa5d9d13", "423d4398", "4484b485", "8ddf9faa"]:
    loc = [name for name, s in [("train", train_df), ("val", val_df), ("test", test_df)]
           if (s["true_group"] == g).any()]
    print(g, "->", loc)

aa5d9d13 -> ['val']
423d4398 -> ['train']
4484b485 -> ['test']
8ddf9faa -> ['train']


In [18]:
def head_tail_truncate_ids(text, tokenizer, max_length=512):
    """Tokenize; if over limit, keep first half and last half of tokens."""
    ids = tokenizer.encode(text, add_special_tokens=False)
    budget = max_length - 2  # room for [CLS]/[SEP]
    if len(ids) <= budget:
        return text
    head = budget // 2
    tail = budget - head
    return tokenizer.decode(ids[:head] + ids[-tail:])

for split in (train_df, val_df, test_df):
    split["text_truncated"] = split["Trace Content"].apply(
        lambda t: head_tail_truncate_ids(t, tokenizer)
    )

# verify: nothing should exceed 512 after re-tokenizing with special tokens
check = test_df["text_truncated"].apply(lambda t: len(tokenizer.encode(t)))
print("max tokens after truncation:", check.max())
print("traces that were truncated (train):",
      (train_df["Trace Content"] != train_df["text_truncated"]).sum(), "/", len(train_df))

max tokens after truncation: 512
traces that were truncated (train): 56 / 259


In [19]:
from datasets import Dataset

def to_hf(split_df):
    return Dataset.from_pandas(
        split_df[["text_truncated", "label_id"]].rename(
            columns={"text_truncated": "text", "label_id": "label"}
        )
    )

train_ds, val_ds, test_ds = to_hf(train_df), to_hf(val_df), to_hf(test_df)

def tokenize_fn(batch):
    return tokenizer(batch["text"], truncation=True, max_length=512, padding=False)

train_ds = train_ds.map(tokenize_fn, batched=True)
val_ds = val_ds.map(tokenize_fn, batched=True)
test_ds = test_ds.map(tokenize_fn, batched=True)

print(train_ds)

Map:   0%|          | 0/259 [00:00<?, ? examples/s]

Map:   0%|          | 0/87 [00:00<?, ? examples/s]

Map:   0%|          | 0/87 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 259
})


In [20]:
import torch
import numpy as np
from torch import nn
from transformers import (AutoModelForSequenceClassification, TrainingArguments,
                          Trainer, DataCollatorWithPadding, EarlyStoppingCallback)
from sklearn.metrics import f1_score, accuracy_score
from sklearn.utils.class_weight import compute_class_weight

num_labels = len(label2id)

class_weights = compute_class_weight(
    "balanced", classes=np.arange(num_labels), y=train_df["label_id"].values
)
class_weights_t = torch.tensor(class_weights, dtype=torch.float32).to("cuda")
print("Class weights:", dict(zip(label2id.keys(), class_weights.round(3))))

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = nn.CrossEntropyLoss(weight=class_weights_t)(
            outputs.logits.view(-1, num_labels), labels.view(-1)
        )
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
        "f1_weighted": f1_score(labels, preds, average="weighted"),
    }

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=num_labels,
    id2label=id2label, label2id=label2id,
)

args = TrainingArguments(
    output_dir="../classifier/checkpoints",
    num_train_epochs=8,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    save_total_limit=2,
    logging_steps=10,
    fp16=True,
    report_to="none",
    seed=42,
)

trainer = WeightedTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

trainer.train()

Class weights: {'HALLUCINATION': np.float64(0.996), 'LOOP': np.float64(1.349), 'SUCCESS': np.float64(0.841), 'UNSAFE_EXECUTION': np.float64(0.938)}


pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

e:\Projects\agent-failure-detection\venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\beeya\.cache\huggingface\hub\models--microsoft--deberta-v3-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


ValueError: Due to a serious vulnerability issue in `torch.load`, even with `weights_only=True`, we now require users to upgrade torch to at least v2.6 in order to use the function. This version restriction does not apply when loading files with safetensors.
See the vulnerability report here https://nvd.nist.gov/vuln/detail/CVE-2025-32434